# FiNER-ORD NER Fine-Tuning DISTILLED-BERT-BASE-CASED



In [1]:
!pip install evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e1b24f99c809ca0477e00f901e7007460d554613c9b4967eda2435acbc89da7f
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


Deps

In [47]:
from datasets import load_dataset, Dataset, DatasetDict
from collections import defaultdict
from transformers import AutoTokenizer
import numpy as np
import evaluate


from transformers import DataCollatorForTokenClassification, AutoModelForTokenClassification, TrainingArguments, Trainer

FiNER-ORD contains rows of isolated tokenss

In [14]:
findata = load_dataset("gtfintechlab/finer-ord")
print(findata)

DatasetDict({
    train: Dataset({
        features: ['gold_label', 'gold_token', 'doc_idx', 'sent_idx'],
        num_rows: 80531
    })
    validation: Dataset({
        features: ['gold_label', 'gold_token', 'doc_idx', 'sent_idx'],
        num_rows: 10233
    })
    test: Dataset({
        features: ['gold_label', 'gold_token', 'doc_idx', 'sent_idx'],
        num_rows: 25957
    })
})


In [15]:
print(findata["train"][0])
print(findata["train"][1])
print(findata["train"][2])
print(findata["train"][3])

{'gold_label': 0, 'gold_token': 'Kenyan', 'doc_idx': 0, 'sent_idx': 0}
{'gold_label': 0, 'gold_token': 'Firms', 'doc_idx': 0, 'sent_idx': 0}
{'gold_label': 0, 'gold_token': 'Eye', 'doc_idx': 0, 'sent_idx': 0}
{'gold_label': 0, 'gold_token': 'Deals', 'doc_idx': 0, 'sent_idx': 0}


In [9]:
label_list = ["O", "PER_B", "PER_I", "LOC_B", "LOC_I", "ORG_B", "ORG_I"]
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}
num_labels = len(label_list)

Un-Isolate tokens into sentences

In [31]:
def group_into_sentences(flat_split):
    grouped = defaultdict(lambda: {"tokens": [], "ner_tags": []})
    for row in flat_split:
        tok = row["gold_token"]
        if tok is None: #after parsing, 1 token was None, so add this
            continue
        tok = str(tok)
        if tok == "":
            continue
        key = (row["doc_idx"], row["sent_idx"])
        grouped[key]["tokens"].append(tok)
        grouped[key]["ner_tags"].append(row["gold_label"])
    return [grouped[k] for k in sorted(grouped.keys()) if len(grouped[k]["tokens"]) > 0]

In [32]:
train = group_into_sentences(findata["train"])
val = group_into_sentences(findata["validation"])
test = group_into_sentences(findata["test"])

print(f"#of train sentences: {len(train)}")
print(f"#of val sentences:   {len(val)}")
print(f"#of test sentences:  {len(test)}")
print(train[0])

#of train sentences: 3262
#of val sentences:   402
#of test sentences:  1075
{'tokens': ['Kenyan', 'Firms', 'Eye', 'Deals', 'During', 'Obama', 'Summit', 'Tagged', ':', 'The', 'Global', 'Entrepreneurship', 'Summit', ',', 'launched', 'by', 'President', 'Obama', 'in', '2009', ',', 'brings', 'together', 'entrepreneurs', 'and', 'investors', 'from', 'across', 'Africa', 'and', 'around', 'the', 'world', 'annually', 'to', 'showcase', 'innovative', 'projects', ',', 'exchange', 'new', 'ideas', ',', 'and', 'help', 'spur', 'economic', 'opportunity', '.'], 'ner_tags': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [33]:
findatasentences = DatasetDict({
    "train":      Dataset.from_list(train),
    "validation": Dataset.from_list(val),
    "test":       Dataset.from_list(test),
})
findatasentences

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3262
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 402
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1075
    })
})

Align word tokens to subword tokens

In [26]:
MODEL_NAME = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [117]:
def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=tokenizer.model_max_length,
    )

    aligned_labels = []

    for batch_idx, label_seq in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=batch_idx)
        prev_word_id = None
        new_labels = []

        for word_id in word_ids:
            if word_id is None:
                new_labels.append(-100)
            elif word_id != prev_word_id:
                new_labels.append(label_seq[word_id])
            else:
                new_labels.append(-100)
            prev_word_id = word_id

        aligned_labels.append(new_labels)

    tokenized["labels"] = aligned_labels
    return tokenized

In [103]:
tokenized_findata = findatasentences.map(
    tokenize_and_align,
    batched=True,
    remove_columns=findatasentences["train"].column_names,
)

Map:   0%|          | 0/3262 [00:00<?, ? examples/s]

Map:   0%|          | 0/402 [00:00<?, ? examples/s]

Map:   0%|          | 0/1075 [00:00<?, ? examples/s]

In [104]:
print(tokenized_findata)
print()
print(tokenized_findata["train"][0])

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3262
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 402
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1075
    })
})

{'input_ids': [101, 22336, 17355, 19995, 9329, 15361, 1116, 1507, 7661, 11843, 9697, 3660, 131, 1109, 5357, 13832, 7877, 1643, 16717, 7719, 3157, 11843, 117, 2536, 1118, 1697, 7661, 1107, 1371, 117, 7100, 1487, 21607, 1105, 9660, 1121, 1506, 2201, 1105, 1213, 1103, 1362, 6089, 1106, 17045, 10034, 3203, 117, 3670, 1207, 4133, 117, 1105, 1494, 16650, 2670, 3767, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': 

data collator

In [105]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval, change FiNER-ORD labels to BIO labels

In [106]:
seqeval = evaluate.load("seqeval")

In [ ]:
def to_bio(label):
    if label == "O":
        return "O"
    typ, pos = label.split("_")
    return f"{pos}-{typ}"

In [116]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [to_bio(id2label[l]) for l in label_seq if l != -100]
        for label_seq in labels
    ]

    true_preds = [
        [to_bio(id2label[p]) for p, l in zip(pred_seq, label_seq) if l != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

Training DISTILLED-BERT-BASE-CASED

In [109]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [110]:
args = TrainingArguments(
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    seed=2,
)

In [111]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_findata["train"],
    eval_dataset=tokenized_findata["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [112]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.250581,0.117768,0.654631,0.684729,0.669342,0.965891
2,0.056931,0.087758,0.725732,0.773399,0.748808,0.974687
3,0.036647,0.085354,0.766773,0.788177,0.777328,0.976251
4,0.026711,0.076974,0.789969,0.827586,0.808340,0.978987
5,0.023245,0.077608,0.783489,0.825944,0.804157,0.978890


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


In [113]:
print(train_result.metrics)

{'train_runtime': 113.0077, 'train_samples_per_second': 144.326, 'train_steps_per_second': 9.026, 'total_flos': 273868551185592.0, 'train_loss': 0.07882298581740435, 'epoch': 5.0}


In [115]:
test_metrics = trainer.evaluate(tokenized_findata["test"])
print("using TESTSET:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")

using TESTSET:
eval_loss: 0.0766
eval_precision: 0.6971
eval_recall: 0.7700
eval_f1: 0.7317
eval_accuracy: 0.9771
eval_runtime: 2.4885
eval_samples_per_second: 431.9840
eval_steps_per_second: 54.2490
epoch: 5.0000
